# Integrating sbi-trained likelihoods into HSSM (NLE + NRE via ONNX)

This tutorial mirrors the structure of `bayesflow_lre_integration.ipynb` for the
third major SBI library in the HSSM ecosystem: **[sbi](https://github.com/sbi-dev/sbi)**
(mackelab). It demonstrates how to:

1. Train a neural likelihood estimator (NLE) on synthetic DDM simulations using sbi.
2. Export the trained estimator to ONNX via
   [`lanfactory.onnx.transform_sbi_to_onnx`](https://alexanderfengler.github.io/LANfactory/exporting_sbi_models/).
3. Load the ONNX file into HSSM exactly like any other LAN-style approximator and run
   MCMC inference.
4. Repeat the loop with a neural ratio estimator (NRE) for comparison.

The ecosystem's current scope is **neural likelihood surrogates** (NLE and NRE). NPE/
posterior-amortized methods are deliberately out of scope here — they don't compose
cleanly with PyMC priors. See the
[Exporting sbi Models guide](https://alexanderfengler.github.io/LANfactory/exporting_sbi_models/)
for the full supported-architecture matrix.

> **Environment note:** This tutorial requires both `hssm` and `lanfactory[all]` (which
> pulls `sbi` and `nflows`) in the same environment. JAX/flax/numpyro pins must be
> resolved jointly across the two packages.

## Part 1 — Setup

In [ ]:
# Enable x64 BEFORE any other JAX-touching import.
# HSSM's onnx2jax auto-flips this when loading sbi-exported ONNX graphs that carry
# int64 tensors (typical for normalizing flows from torch.onnx.export), but setting
# it explicitly here is best practice and silences the auto-flip UserWarning.
import jax
jax.config.update("jax_enable_x64", True)

from pathlib import Path
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

import hssm
from sbi.inference import NLE_A, NRE_A
from sbi.neural_nets import classifier_nn
from sbi.utils import BoxUniform
from ssms.basic_simulators.simulator import simulator

# Import transform_sbi_to_onnx with a fallback. The clean path works when
# lanfactory is installed in an env whose JAX/flax pins are compatible with this
# notebook's other deps. Until the cross-repo env alignment lands, the fallback
# loads only the sbi exporter module directly, bypassing lanfactory's top-level
# __init__.py (which would otherwise pull the flax-dependent jax_mlp trainer).
try:
    from lanfactory.onnx import transform_sbi_to_onnx
except ImportError:
    import importlib.util
    import os

    # Try candidates: explicit env var, then several relative paths covering
    # common Jupyter launch contexts (notebook dir / repo root / spine root).
    _candidates = [
        os.environ.get("LANFACTORY_SBI_PATH"),
        "../../../LANfactory/src/lanfactory/onnx/sbi.py",  # cwd = notebook dir
        "repos/LANfactory/src/lanfactory/onnx/sbi.py",     # cwd = HSSMSpine root
        "../LANfactory/src/lanfactory/onnx/sbi.py",        # cwd = repos/HSSM root
    ]
    _path = None
    for _c in _candidates:
        if _c and Path(_c).exists():
            _path = Path(_c).resolve()
            break
    if _path is None:
        raise ImportError(
            "Could not locate lanfactory/onnx/sbi.py. Set the LANFACTORY_SBI_PATH "
            "environment variable to the absolute path of that file, or run the "
            "notebook from a directory where one of the relative candidates resolves: "
            f"{[c for c in _candidates if c]}"
        )
    _spec = importlib.util.spec_from_file_location("_lanfactory_sbi", _path)
    _mod = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    transform_sbi_to_onnx = _mod.transform_sbi_to_onnx
    print(f"(fallback) loaded transform_sbi_to_onnx from {_path}")

np.random.seed(0)
torch.manual_seed(0)

# Training budget. Bumped from the original 10k/50ep because the wider prior
# (matching HSSM's default DDM bounds — see Part 3) needs more samples to
# cover the parameter volume.
N_TRAIN = 30_000   # training simulations
N_OBS = 500        # observed trials at the true theta
NUM_EPOCHS = 100
MCMC_DRAWS = 500
MCMC_TUNE = 500
MCMC_CHAINS = 2

## Part 2 — Simulate observed DDM data

We use the standard 4-parameter DDM (`v`, `a`, `z`, `t`) from `ssm-simulators`. The
true parameters and parameter ranges below match the BayesFlow LRE tutorial so that
posteriors from the two tutorials can be compared apples-to-apples.

In [ ]:
DDM_PARAM_NAMES = ["v", "a", "z", "t"]
# Training prior matches HSSM's default bounds for model="ddm" with
# loglik_kind="approx_differentiable" — see hssm.defaults.default_model_config.
# This is important: if the training prior is narrower than HSSM's posterior
# can explore, MCMC will walk into parameter regions the flow never saw and
# extrapolate badly (spurious likelihood pockets, biased posteriors).
PRIOR_LOW = np.array([-3.0, 0.3, 0.0, 0.0], dtype=np.float32)
PRIOR_HIGH = np.array([3.0, 2.5, 1.0, 2.0], dtype=np.float32)
TRUE_THETA = np.array([0.5, 1.2, 0.5, 0.25], dtype=np.float32)

out = simulator(theta=TRUE_THETA[None, :], model="ddm", n_samples=N_OBS)
obs_data = pd.DataFrame(
    {
        "rt": out["rts"].squeeze().astype(np.float32),
        "response": out["choices"].squeeze().astype(np.float32),
    }
)
print(f"observed: {len(obs_data)} trials at true theta = {TRUE_THETA}")
obs_data.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for resp, color in zip([-1, 1], ["C0", "C1"]):
    mask = obs_data["response"] == resp
    ax.hist(
        obs_data.loc[mask, "rt"],
        bins=40,
        alpha=0.6,
        label=f"choice={int(resp)}",
        color=color,
    )
ax.set_xlabel("RT (s)")
ax.set_ylabel("count")
ax.set_title("Observed RT histogram by choice")
ax.legend()
plt.tight_layout()
plt.show()

## Part 3 — Train a sbi NLE_A on DDM simulations

`NLE_A` trains a conditional density estimator (here a MAF normalizing flow) on
`(theta, x)` pairs. After training, `estimator.log_prob(x, condition=theta)` returns
`log p(x | theta)` with z-score standardization Jacobians applied automatically.

In [ ]:
prior = BoxUniform(
    low=torch.from_numpy(PRIOR_LOW),
    high=torch.from_numpy(PRIOR_HIGH),
)
theta_train = prior.sample((N_TRAIN,))

# Batched simulation: one (rt, choice) per training theta. ssm-simulators
# accepts theta of shape (N, 4) with n_samples=1 and returns rts/choices of
# shape (N, 1) directly — much faster than a Python loop for large N.
sim = simulator(
    theta=theta_train.numpy().astype(np.float32),
    model="ddm",
    n_samples=1,
)
x_train = torch.from_numpy(
    np.stack([sim["rts"].squeeze(-1), sim["choices"].squeeze(-1)], axis=-1).astype(
        np.float32
    )
)
print(f"training set: theta={theta_train.shape}, x={x_train.shape}")

In [ ]:
inference_nle = NLE_A(prior=prior, density_estimator="maf")
estimator_nle = inference_nle.append_simulations(theta_train, x_train).train(
    training_batch_size=200,
    max_num_epochs=NUM_EPOCHS,
)
estimator_nle.eval()
print("NLE training complete")

## Part 4 — Export the trained NLE to ONNX

The exporter wraps `estimator.log_prob` as a `torch.nn.Module` whose `forward(combined)`
splits a concatenated `(theta, x)` input. The result is a single-trial ONNX graph that
HSSM consumes exactly like a LAN file.

In [ ]:
onnx_dir = Path("./sbi_onnx_artifacts")
onnx_dir.mkdir(exist_ok=True)
nle_onnx_path = onnx_dir / "ddm_nle.onnx"

transform_sbi_to_onnx(
    estimator_nle,
    str(nle_onnx_path),
    mode="nle",
    example_theta_dim=4,
    example_x_dim=2,
)
print(f"exported NLE: {nle_onnx_path}  ({nle_onnx_path.stat().st_size:,} bytes)")

## Part 5 — High-level integration via `hssm.HSSM()`

HSSM's `loglik_kind="approx_differentiable"` path consumes the `.onnx` file
identically to a LAN-trained network. With `model="ddm"` HSSM already knows the
parameter list and response columns; we just hand it the file.

In [ ]:
model_nle = hssm.HSSM(
    data=obs_data,
    model="ddm",
    loglik_kind="approx_differentiable",
    loglik=str(nle_onnx_path),
    p_outlier=0,
)
print(model_nle)

In [ ]:
idata_nle = model_nle.sample(
    sampler="numpyro",
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=MCMC_CHAINS,
    target_accept=0.9,
    progressbar=False,
)

In [ ]:
summary_nle = az.summary(idata_nle, var_names=DDM_PARAM_NAMES)
summary_nle

In [ ]:
az.plot_trace(idata_nle, var_names=DDM_PARAM_NAMES)
plt.tight_layout()
plt.show()

### Part 5b — Diagnostic: is the NLE itself biased, or is HSSM not finding its mode?

If the marginal posteriors miss the truth, two very different causes are possible:

1. **Training quality** — the flow itself is maximized at the wrong θ.
2. **Sampling / HSSM-side** — the flow is fine, but MCMC isn't finding its mode (priors, init, mixing).

We can distinguish them by asking the trained NLE directly: which θ does it
think makes the observed data more likely — the true θ, or the posterior's
marginal mean? If the NLE *itself* prefers the (wrong) posterior mean, training
is the issue. If it prefers the truth, sampling is.

In [ ]:
posterior_mean_nle = {
    p: float(idata_nle.posterior[p].mean()) for p in DDM_PARAM_NAMES
}
obs_x_t = torch.from_numpy(
    obs_data[["rt", "response"]].values.astype(np.float32)
)

def total_logp(estimator, theta_dict):
    """Sum log p(x_i | theta) over all observed trials at a single theta."""
    theta_row = torch.tensor(
        [[theta_dict[p] for p in DDM_PARAM_NAMES]], dtype=torch.float32
    ).repeat(len(obs_x_t), 1)
    with torch.no_grad():
        return estimator.log_prob(obs_x_t, condition=theta_row).sum().item()

lp_true = total_logp(estimator_nle, dict(zip(DDM_PARAM_NAMES, TRUE_THETA)))
lp_mean = total_logp(estimator_nle, posterior_mean_nle)

print(f"NLE log-prob at true theta:      {lp_true:+.2f}")
print(f"NLE log-prob at posterior mean:  {lp_mean:+.2f}")
print(f"Δ (mean − true):                 {lp_mean - lp_true:+.2f}")
print()
if lp_mean > lp_true + 5.0:
    print("→ NLE itself prefers the wrong θ by a large margin.")
    print("  Diagnosis: TRAINING QUALITY. The flow has not learned the true")
    print("  conditional density well. Fix with more sims, larger model, or")
    print("  more epochs.")
elif lp_mean > lp_true:
    print("→ NLE mildly prefers the posterior mean over the truth.")
    print("  Diagnosis: marginal posterior bias may reflect a slightly")
    print("  miscalibrated flow plus narrow likelihood. Marginal posteriors")
    print("  can shift while the joint mode stays near the truth — check the")
    print("  pairs plot (added separately) to see the joint.")
else:
    print("→ NLE prefers the truth; the wrong posterior mean is a sampling")
    print("  artifact (priors / init / mixing), not a training issue.")
print()
print(f"Posterior mean: {posterior_mean_nle}")
print(f"True theta:     {dict(zip(DDM_PARAM_NAMES, TRUE_THETA.tolist()))}")

## Part 6 — Brief NRE variant

The same pipeline works for ratio classifiers. The only sbi-side wrinkle is that the
default MLP classifier uses `nn.LayerNorm` between hidden layers, and `jaxonnxruntime`
doesn't implement `LayerNormalization`. We pass `norm_layer=nn.Identity` to disable
it. See `docs/exporting_sbi_models.md` in LANfactory for the full constraint list.

In [ ]:
classifier_builder = classifier_nn(model="mlp", norm_layer=nn.Identity)
inference_nre = NRE_A(prior=prior, classifier=classifier_builder)
classifier_nre = inference_nre.append_simulations(theta_train, x_train).train(
    training_batch_size=200,
    max_num_epochs=NUM_EPOCHS,
)
classifier_nre.eval()
print("NRE training complete")

In [ ]:
nre_onnx_path = onnx_dir / "ddm_nre.onnx"
transform_sbi_to_onnx(
    classifier_nre,
    str(nre_onnx_path),
    mode="nre",
    example_theta_dim=4,
    example_x_dim=2,
)
print(f"exported NRE: {nre_onnx_path}  ({nre_onnx_path.stat().st_size:,} bytes)")

model_nre = hssm.HSSM(
    data=obs_data,
    model="ddm",
    loglik_kind="approx_differentiable",
    loglik=str(nre_onnx_path),
    p_outlier=0,
)
idata_nre = model_nre.sample(
    sampler="numpyro",
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=MCMC_CHAINS,
    target_accept=0.9,
    progressbar=False,
)
summary_nre = az.summary(idata_nre, var_names=DDM_PARAM_NAMES)
summary_nre

## Part 7 — Posterior comparison: NLE vs NRE vs ground truth

The comparison plot is the deliverable. NLE and NRE should cover the true parameter
values; both posteriors should be tighter than the prior and centred near the truth.
For a four-way comparison alongside the LAN baseline and BayesFlow-LRE result on the
same simulated data, load their cached posteriors here and add panels to the plot.

> Reuses the exact `TRUE_THETA`, `N_OBS`, and prior ranges from
> `bayesflow_lre_integration.ipynb`, so the BayesFlow-LRE posteriors are directly
> comparable when added.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name, true_val in zip(axes, DDM_PARAM_NAMES, TRUE_THETA):
    samples_nle = idata_nle.posterior[name].values.flatten()
    samples_nre = idata_nre.posterior[name].values.flatten()
    ax.hist(samples_nle, bins=30, alpha=0.5, label="sbi NLE", color="C0", density=True)
    ax.hist(samples_nre, bins=30, alpha=0.5, label="sbi NRE", color="C1", density=True)
    ax.axvline(true_val, color="red", linestyle="--", linewidth=2, label="true")
    ax.set_xlabel(name)
    ax.set_title(f"posterior over {name}")
    ax.legend()
fig.suptitle("DDM posterior recovery: sbi NLE vs NRE", y=1.02)
plt.tight_layout()
plt.show()

## Summary

We trained a sbi NLE_A and an NRE_A on synthetic DDM data, exported both to ONNX via
`lanfactory.onnx.transform_sbi_to_onnx`, and ran MCMC through HSSM's existing
`loglik_kind="approx_differentiable"` pipeline. Both posteriors recover the true
parameters.

**Where to look next:**
- LANfactory's [Exporting sbi Models guide](https://alexanderfengler.github.io/LANfactory/exporting_sbi_models/) — supported-architecture matrix, known constraints, troubleshooting.
- The BayesFlow LRE tutorial (`bayesflow_lre_integration.ipynb`) — the same DDM with a different SBI library, for cross-toolkit comparison.
- LAN tutorials (`main_tutorial.ipynb`) — the original LANfactory workflow this integration builds on top of.

**Known constraints (v1):**
- sbi NPE/SNPE estimators are deliberately out of scope (posterior-shaped, conflicts with PyMC priors).
- Neural Spline Flows are blocked on a missing `SearchSorted` op in `jaxonnxruntime`.
- FMPE / NPSE (score-based) require ODE integration and aren't ONNX-exportable.